# 🍽️ トレー種類 × 食品残渣 マルチタスク分類モデル

EfficientNet-B0 の転移学習で、1つのモデルから
- **トレーの種類**（複数種類対応）
- **clean（残渣なし） / dirty（残渣あり）**

を同時に予測します。

---
### 📋 使い方（3ステップ）
1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** に設定
2. `data.zip` を用意して Step 2 でアップロード
3. あとはセルを上から順に実行するだけ

### 📁 データの準備方法
ZIP の中身を以下のフォルダ構造にしてください。
**フォルダ名は必ず `<トレー種類>_clean` / `<トレー種類>_dirty` の形式**にします。
```
data/
├── train/
│   ├── trayA_clean/
│   ├── trayA_dirty/
│   ├── trayB_clean/
│   ├── trayB_dirty/
│   └── ... (トレーの種類ぶん繰り返す)
└── val/
    ├── trayA_clean/
    ├── trayA_dirty/
    └── ...
```
トレーの種類数や名前は自由（例: `donburi`, `bento`, `set_a` など日本語以外の名前を推奨）。

## ⚙️ Step 1: 環境セットアップ

In [ ]:
import torch, torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'torchvision : {torchvision.__version__}')
print(f'GPU 使用可能 : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

In [ ]:
import os, copy, random, zipfile, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {DEVICE}')

IMG_SIZE = 224
CLEANLINESS_NAMES = ['clean', 'dirty']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

## 📤 Step 2: data.zip をアップロード

In [ ]:
from google.colab import files

print('data.zip を選択してください')
uploaded = files.upload()

# 既存の data/ を削除してからクリーンに展開
shutil.rmtree('data', ignore_errors=True)

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
print('展開完了\n')

for split in ['train', 'val']:
    print(f'[{split}]')
    for folder in sorted(Path(f'data/{split}').iterdir()):
        if folder.is_dir():
            n = len(list(folder.glob('*')))
            print(f'  {folder.name}: {n} 枚')

## 📐 Step 3: データセット定義

In [ ]:
class MultiTaskTrayDataset(Dataset):
    def __init__(self, root_dir, transform=None, tray_type_names=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        parsed = []
        for folder in sorted(self.root_dir.iterdir()):
            if not folder.is_dir():
                continue
            tray_type, cleanliness = self._parse_folder_name(folder.name)
            for img_path in sorted(folder.glob('*')):
                if img_path.suffix.lower() in IMAGE_EXTS:
                    parsed.append((img_path, tray_type, cleanliness))

        if not parsed:
            raise ValueError(f'画像が見つかりません: {self.root_dir}')

        if tray_type_names is None:
            tray_type_names = sorted({t for _, t, _ in parsed})
        self.tray_type_names = tray_type_names
        tray_type_to_idx = {name: i for i, name in enumerate(tray_type_names)}

        self.samples = []
        for img_path, tray_type, cleanliness in parsed:
            if tray_type not in tray_type_to_idx:
                raise ValueError(
                    f"未知のトレー種類です: '{tray_type}' ({img_path})\n"
                    f'既知のトレー種類: {tray_type_names}'
                )
            tray_idx = tray_type_to_idx[tray_type]
            clean_idx = CLEANLINESS_NAMES.index(cleanliness)
            self.samples.append((img_path, tray_idx, clean_idx))

    @staticmethod
    def _parse_folder_name(name):
        for suffix in CLEANLINESS_NAMES:
            token = f'_{suffix}'
            if name.endswith(token):
                tray_type = name[:-len(token)]
                if not tray_type:
                    raise ValueError(f"フォルダ名が不正です: '{name}'")
                return tray_type, suffix
        raise ValueError(
            f"フォルダ名は '<トレー種類>_clean' か '<トレー種類>_dirty' にしてください: '{name}'"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, tray_idx, clean_idx = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, tray_idx, clean_idx

In [ ]:
# データ拡張・正規化
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = MultiTaskTrayDataset('data/train', train_tf)
val_ds   = MultiTaskTrayDataset('data/val', val_tf, tray_type_names=train_ds.tray_type_names)

TRAY_TYPE_NAMES = train_ds.tray_type_names
print(f'トレー種類({len(TRAY_TYPE_NAMES)}): {TRAY_TYPE_NAMES}')
print(f'train: {len(train_ds)} 枚 / val: {len(val_ds)} 枚')

## 🔧 Step 4: 学習設定

In [ ]:
# ========== 変更したい場合はここを編集 ==========
BATCH_SIZE         = 16    # データが少ない場合は 8 に下げる
EPOCHS             = 30
LR                 = 1e-3
UNFREEZE_AT        = 15    # このエポックから全層 fine-tune
CLEANLINESS_WEIGHT = 1.0   # clean/dirty 判定を重視したい場合は大きくする
# ================================================

train_batch_size = min(BATCH_SIZE, len(train_ds))
# BatchNorm はバッチサイズ1だと学習時にエラーになるため、余りがちょうど1枚になる場合のみ最終バッチを捨てる
drop_last = train_batch_size > 1 and len(train_ds) % train_batch_size == 1

dataloaders = {
    'train': DataLoader(train_ds, batch_size=train_batch_size, shuffle=True, num_workers=2, drop_last=drop_last),
    'val'  : DataLoader(val_ds, batch_size=min(BATCH_SIZE, len(val_ds)), shuffle=False, num_workers=2),
}
print(f'バッチサイズ: train={train_batch_size}(drop_last={drop_last}) / エポック数: {EPOCHS} / 学習率: {LR}')

## 🧠 Step 5: モデル定義

In [ ]:
class MultiTaskTrayClassifier(nn.Module):
    def __init__(self, num_tray_types, freeze_base=True):
        super().__init__()
        backbone = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        in_features = backbone.classifier[1].in_features

        if freeze_base:
            for p in self.features.parameters():
                p.requires_grad = False

        self.dropout = nn.Dropout(p=0.3)
        self.tray_type_head = nn.Linear(in_features, num_tray_types)
        self.cleanliness_head = nn.Linear(in_features, len(CLEANLINESS_NAMES))

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.tray_type_head(x), self.cleanliness_head(x)

    def unfreeze_backbone(self):
        for p in self.features.parameters():
            p.requires_grad = True

model = MultiTaskTrayClassifier(num_tray_types=len(TRAY_TYPE_NAMES), freeze_base=True).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'学習パラメータ: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

## 🏋️ Step 6: 学習

In [ ]:
def run_epoch(model, loader, criterion, optimizer, train):
    model.train() if train else model.eval()
    total_loss, tray_correct, clean_correct, n = 0.0, 0, 0, 0

    for inputs, tray_labels, clean_labels in loader:
        inputs = inputs.to(DEVICE)
        tray_labels = tray_labels.to(DEVICE)
        clean_labels = clean_labels.to(DEVICE)

        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            tray_logits, clean_logits = model(inputs)
            loss = criterion(tray_logits, tray_labels) \
                + CLEANLINESS_WEIGHT * criterion(clean_logits, clean_labels)
            if train:
                loss.backward()
                optimizer.step()

        bs = inputs.size(0)
        total_loss += loss.item() * bs
        tray_correct += (tray_logits.argmax(1) == tray_labels).sum().item()
        clean_correct += (clean_logits.argmax(1) == clean_labels).sum().item()
        n += bs

    return {'loss': total_loss/n, 'tray_acc': tray_correct/n, 'clean_acc': clean_correct/n}


os.makedirs('checkpoints', exist_ok=True)
criterion = nn.CrossEntropyLoss()
head_params = list(model.tray_type_head.parameters()) + list(model.cleanliness_head.parameters())
optimizer = optim.Adam(head_params, lr=LR)
phase1_epochs = min(UNFREEZE_AT, EPOCHS)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase1_epochs)

best_acc = 0.0
best_wts = copy.deepcopy(model.state_dict())
history = {'train_loss':[], 'train_tray_acc':[], 'train_clean_acc':[],
           'val_loss':[], 'val_tray_acc':[], 'val_clean_acc':[]}

for epoch in range(1, EPOCHS + 1):
    if epoch == phase1_epochs + 1:
        print('\n=== Phase 2: 全層 fine-tune 開始 ===')
        model.unfreeze_backbone()
        optimizer = optim.Adam(model.parameters(), lr=LR * 0.1)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - phase1_epochs)

    train_m = run_epoch(model, dataloaders['train'], criterion, optimizer, True)
    val_m   = run_epoch(model, dataloaders['val'],   criterion, optimizer, False)

    print(f'Epoch {epoch:>3}/{EPOCHS}  '
          f"train loss={train_m['loss']:.3f} tray={train_m['tray_acc']:.3f} clean={train_m['clean_acc']:.3f}  |  "
          f"val loss={val_m['loss']:.3f} tray={val_m['tray_acc']:.3f} clean={val_m['clean_acc']:.3f}", end='')

    for k, v in [('train_loss', train_m['loss']), ('train_tray_acc', train_m['tray_acc']),
                  ('train_clean_acc', train_m['clean_acc']), ('val_loss', val_m['loss']),
                  ('val_tray_acc', val_m['tray_acc']), ('val_clean_acc', val_m['clean_acc'])]:
        history[k].append(v)

    combined_acc = (val_m['tray_acc'] + val_m['clean_acc']) / 2
    if combined_acc > best_acc:
        best_acc = combined_acc
        best_wts = copy.deepcopy(model.state_dict())
        torch.save({'model_state_dict': best_wts, 'tray_type_names': TRAY_TYPE_NAMES},
                   'checkpoints/best_model.pth')
        print('  ✅ saved', end='')
    print()
    scheduler.step()

model.load_state_dict(best_wts)
print(f'\n🎉 学習完了  Best combined val acc: {best_acc:.4f}')

## 📊 Step 7: 学習曲線の確認

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, history['train_loss'], label='train')
axes[0].plot(epochs_range, history['val_loss'],   label='val')
axes[0].axvline(x=phase1_epochs, color='gray', linestyle='--', label='fine-tune開始')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_range, history['train_tray_acc'], label='train')
axes[1].plot(epochs_range, history['val_tray_acc'],   label='val')
axes[1].axvline(x=phase1_epochs, color='gray', linestyle='--', label='fine-tune開始')
axes[1].set_title('Tray Type Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(epochs_range, history['train_clean_acc'], label='train')
axes[2].plot(epochs_range, history['val_clean_acc'],   label='val')
axes[2].axvline(x=phase1_epochs, color='gray', linestyle='--', label='fine-tune開始')
axes[2].set_title('Cleanliness Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('checkpoints/training_curve.png', dpi=150)
plt.show()

## 📈 Step 8: 評価（混同行列・分類レポート）

In [ ]:
model.eval()
tray_labels, tray_preds, clean_labels, clean_preds = [], [], [], []

with torch.no_grad():
    for inputs, t_labels, c_labels in dataloaders['val']:
        tray_logits, clean_logits = model(inputs.to(DEVICE))
        tray_preds.extend(tray_logits.argmax(1).cpu().numpy())
        clean_preds.extend(clean_logits.argmax(1).cpu().numpy())
        tray_labels.extend(t_labels.numpy())
        clean_labels.extend(c_labels.numpy())

print('=== トレー種類 Classification Report ===')
print(classification_report(tray_labels, tray_preds, target_names=TRAY_TYPE_NAMES, zero_division=0))

print('=== 清潔度 (clean/dirty) Classification Report ===')
print(classification_report(clean_labels, clean_preds, target_names=CLEANLINESS_NAMES, zero_division=0))

cm = confusion_matrix(clean_labels, clean_preds, labels=[0, 1])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(CLEANLINESS_NAMES); ax.set_yticklabels(CLEANLINESS_NAMES)
ax.set_xlabel('予測'); ax.set_ylabel('正解'); ax.set_title('清潔度 混同行列')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center',
               color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=16)
plt.tight_layout()
plt.show()

## 🔍 Step 9: 画像をアップロードして判定する

In [ ]:
from google.colab import files as colab_files

INFER_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(img_path):
    img = Image.open(img_path).convert('RGB')
    tensor = INFER_TF(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        tray_logits, clean_logits = model(tensor)
        tray_probs = F.softmax(tray_logits, dim=1)[0].cpu().numpy()
        clean_probs = F.softmax(clean_logits, dim=1)[0].cpu().numpy()
    tray_idx = tray_probs.argmax()
    clean_idx = clean_probs.argmax()
    return img, TRAY_TYPE_NAMES[tray_idx], tray_probs[tray_idx], CLEANLINESS_NAMES[clean_idx], clean_probs[clean_idx]

print('判定したい画像をアップロードしてください（複数可）')
uploaded = colab_files.upload()

n = len(uploaded)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, fname in zip(axes, uploaded.keys()):
    img, tray_type, tray_p, cleanliness, clean_p = predict(fname)
    color = 'steelblue' if cleanliness == 'clean' else 'tomato'
    mark = '✅' if cleanliness == 'clean' else '⚠️'
    title = f'{mark} {tray_type} ({tray_p:.0%}) / {cleanliness} ({clean_p:.0%})'
    ax.imshow(img)
    ax.set_title(title, color=color, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 💾 Step 10: モデルを保存する

In [ ]:
# Google Drive に保存
from google.colab import drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/tray_classifier_multitask_best_model.pth'
shutil.copy('checkpoints/best_model.pth', save_path)
print(f'Google Drive に保存: {save_path}')

In [ ]:
# ローカルへダウンロード
from google.colab import files as colab_files
colab_files.download('checkpoints/best_model.pth')